# RWH Window/Roof — YOLOv8 Instance Segmentation (Colab)
This notebook trains **YOLOv8-seg** on a Roboflow-exported dataset (**YOLOv8 Segmentation** format), validates it, and saves predictions + metrics.

**Opinionated defaults:** padded resize (already in your Roboflow version), light augmentation (already), and a small model (`yolov8n-seg.pt`) for a stable baseline.


## 1) Setup
Run this once per Colab session.

In [1]:

# === Cell 1: Environment & installs (robust) ===
import os, sys, subprocess, textwrap, platform

def pip_install(pkgs: str):
    print(f"Installing: {pkgs}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs.split())

# Ultralytics is the training engine (YOLOv8/seg)
try:
    import ultralytics  # noqa
except Exception:
    pip_install("ultralytics")

# Small utilities
try:
    import yaml  # noqa
except Exception:
    pip_install("pyyaml")

import torch
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
import ultralytics
print("Ultralytics:", ultralytics.__version__)


Installing: ultralytics
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python: 3.12.12
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
Torch: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
Ultralytics: 8.3.246


## 2) Load your dataset ZIP
Use **one** of the following:
- **A)** Upload the ZIP from your computer (recommended)
- **B)** Put the ZIP in Google Drive and set `DATA_ZIP_PATH`

**Important:** Export from Roboflow as **YOLOv8 Segmentation** (not SAM2).

In [2]:

# === Cell 2: Upload or locate dataset zip ===
from pathlib import Path

# Option B: If your ZIP is in Google Drive, set this path and run.
DATA_ZIP_PATH = ""  # e.g., "/content/drive/MyDrive/RWH_Segmentation_YOLOv8Seg.zip"

# Mount Drive only if you use it
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

zip_path = None
if DATA_ZIP_PATH and Path(DATA_ZIP_PATH).exists():
    zip_path = Path(DATA_ZIP_PATH)
else:
    # Option A: Upload
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) == 0:
        raise FileNotFoundError("No file uploaded. Upload the Roboflow YOLOv8 Segmentation ZIP.")
    zip_path = Path(next(iter(uploaded.keys())))

print("Using ZIP:", zip_path)


Saving RWH Segmentation.v1-1.yolov8.zip to RWH Segmentation.v1-1.yolov8.zip
Using ZIP: RWH Segmentation.v1-1.yolov8.zip


## 3) Unzip + find `data.yaml`
This cell unzips to `/content/dataset` and patches the YAML paths so training is error-free.

In [3]:

# === Cell 3: Unzip, find yaml, patch paths ===
import zipfile, shutil, yaml, os
from pathlib import Path

OUT_DIR = Path("/content/dataset")
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(OUT_DIR)

# Find a likely dataset root that contains data.yaml
yaml_candidates = list(OUT_DIR.rglob("data.yaml")) + list(OUT_DIR.rglob("dataset.yaml"))
if not yaml_candidates:
    # Some exports name it differently; fallback to any yaml containing 'train:' and 'val:'
    yaml_candidates = list(OUT_DIR.rglob("*.yaml")) + list(OUT_DIR.rglob("*.yml"))

def looks_like_data_yaml(p: Path) -> bool:
    try:
        d = yaml.safe_load(p.read_text())
        return isinstance(d, dict) and ("train" in d) and ("val" in d) and ("names" in d or "nc" in d)
    except Exception:
        return False

yaml_candidates = [p for p in yaml_candidates if looks_like_data_yaml(p)]
if not yaml_candidates:
    raise FileNotFoundError("Could not find a valid data.yaml. Ensure you exported 'YOLOv8 Segmentation' from Roboflow.")

data_yaml = yaml_candidates[0]
print("Found YAML:", data_yaml)

data = yaml.safe_load(data_yaml.read_text())

# Determine dataset root: Roboflow exports often place images/labels under a folder next to data.yaml
dataset_root = data_yaml.parent

# Normalize paths to be robust
# YOLO expects: path (root), train/val/test relative to that root OR absolute.
data["path"] = str(dataset_root)

def norm_split(x):
    if x is None:
        return None
    # Keep as-is if absolute; otherwise make it relative to dataset_root
    x = str(x)
    if os.path.isabs(x):
        return x
    # Many Roboflow yamls use ../train/images etc relative to yaml location
    abs_path = (dataset_root / x).resolve()
    return str(abs_path)

for k in ["train", "val", "test"]:
    if k in data and data[k] is not None:
        data[k] = norm_split(data[k])

# Write patched yaml
PATCHED_YAML = OUT_DIR / "data_patched.yaml"
PATCHED_YAML.write_text(yaml.safe_dump(data, sort_keys=False))
print("Wrote patched YAML:", PATCHED_YAML)
print(yaml.safe_dump(data, sort_keys=False))


Found YAML: /content/dataset/data.yaml
Wrote patched YAML: /content/dataset/data_patched.yaml
train: /content/train/images
val: /content/valid/images
test: /content/test/images
nc: 2
names:
- Roof
- Window
roboflow:
  workspace: segmentation-3owkj
  project: rwh-segmentation
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/segmentation-3owkj/rwh-segmentation/dataset/1
path: /content/dataset



## 4) Quick label sanity-check (overlay polygons)
This renders a few samples with masks drawn from YOLOv8-seg label files.

In [4]:

# === Cell 4: Visual sanity-check (polygons overlay) ===
import random, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

# Find train images folder from patched yaml
import yaml
cfg = yaml.safe_load(Path(PATCHED_YAML).read_text())
train_images = Path(cfg["train"])
val_images = Path(cfg["val"])

def list_images(folder: Path):
    exts = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
    return [p for p in folder.rglob("*") if p.suffix.lower() in exts]

train_imgs = list_images(train_images)
if len(train_imgs) == 0:
    raise FileNotFoundError(f"No images found under: {train_images}")

# Try to find labels folder in typical YOLO layout
def corresponding_label(img_path: Path) -> Path:
    # Roboflow typically: images/... -> labels/... and same stem
    parts = list(img_path.parts)
    if "images" in parts:
        parts[parts.index("images")] = "labels"
    lbl = Path(*parts).with_suffix(".txt")
    return lbl

names = cfg.get("names", None)
if isinstance(names, dict):
    # convert {0:"roof",1:"window"} to list
    names = [names[i] for i in sorted(names.keys())]
elif not isinstance(names, list):
    names = None

def draw_yolo_seg(img_path: Path):
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    draw = ImageDraw.Draw(img, "RGBA")

    lbl = corresponding_label(img_path)
    if not lbl.exists():
        return img, f"NO LABEL: {lbl.name}"

    lines = [ln.strip() for ln in lbl.read_text().splitlines() if ln.strip()]
    if len(lines) == 0:
        return img, f"EMPTY LABEL: {lbl.name}"

    for ln in lines:
        toks = ln.split()
        cls = int(float(toks[0]))
        pts = list(map(float, toks[1:]))
        if len(pts) < 6 or len(pts) % 2 != 0:
            continue
        xy = [(pts[i]*W, pts[i+1]*H) for i in range(0, len(pts), 2)]
        # semi-transparent fill + outline
        draw.polygon(xy, fill=(0, 255, 0, 70), outline=(0, 255, 0, 200))
        label = str(cls) if names is None or cls >= len(names) else f"{cls}:{names[cls]}"
        # label at first vertex
        x0, y0 = xy[0]
        draw.text((x0+3, y0+3), label, fill=(255, 255, 255, 255))

    return img, lbl.name

# Show a few random samples
k = 3
samples = random.sample(train_imgs, k=min(k, len(train_imgs)))
plt.figure(figsize=(14, 4*k))
for i, p in enumerate(samples, 1):
    vis, tag = draw_yolo_seg(p)
    plt.subplot(k, 1, i)
    plt.imshow(vis)
    plt.title(f"{p.name}  |  {tag}")
    plt.axis("off")
plt.show()


FileNotFoundError: No images found under: /content/train/images

## 5) Train YOLOv8-seg
Baseline training. With 17 real images, do not overthink hyperparameters. Keep it stable.

In [ ]:

# === Cell 5: Train ===
import os
from ultralytics import YOLO

# Choose model size: n (recommended), s, m...
MODEL = "yolov8n-seg.pt"  # small baseline; change to yolov8s-seg.pt later if needed

# Training config (stable defaults)
EPOCHS = 80
IMGSZ = 640
BATCH = -1           # auto
PATIENCE = 30        # early stopping
SEED = 42

model = YOLO(MODEL)

train_results = model.train(
    data=str(PATCHED_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    patience=PATIENCE,
    seed=SEED,
    device=0 if torch.cuda.is_available() else "cpu",
    workers=2,
    verbose=True,
    project="runs",
    name="rwh_yolov8_seg_baseline",
)

print("Training complete. Run directory:", train_results.save_dir if hasattr(train_results, "save_dir") else "runs/rwh_yolov8_seg_baseline")


## 6) Validate + export metrics
This prints and saves key segmentation metrics (mAP50, mAP50-95) and per-class mAP when available.

In [ ]:

# === Cell 6: Validate & metrics ===
import pandas as pd
from pathlib import Path

val_results = model.val(
    data=str(PATCHED_YAML),
    imgsz=IMGSZ,
    device=0 if torch.cuda.is_available() else "cpu",
)

# Robust metric extraction across ultralytics versions
metrics = {}
try:
    # Newer: results_dict
    if hasattr(val_results, "results_dict"):
        metrics.update(val_results.results_dict)
except Exception:
    pass

# Common segmentation fields
def safe_get(obj, path, default=None):
    cur = obj
    for k in path:
        if not hasattr(cur, k):
            return default
        cur = getattr(cur, k)
    return cur

# Try seg metrics
seg = getattr(val_results, "seg", None)
if seg is not None:
    for k in ["map", "map50", "map75", "mp", "mr"]:
        v = getattr(seg, k, None)
        if v is not None:
            metrics[f"seg_{k}"] = float(v)
    maps = getattr(seg, "maps", None)  # per-class mAP
    if maps is not None:
        metrics["seg_maps_per_class"] = [float(x) for x in maps]

# Also include box metrics if present (sometimes)
box = getattr(val_results, "box", None)
if box is not None:
    for k in ["map", "map50", "map75", "mp", "mr"]:
        v = getattr(box, k, None)
        if v is not None:
            metrics[f"box_{k}"] = float(v)

print("=== Validation metrics ===")
for k, v in metrics.items():
    if k.endswith("per_class"):
        print(k, ":", v)
    else:
        print(f"{k}: {v}")

# Save a tidy CSV
run_dir = Path(getattr(val_results, "save_dir", "runs/rwh_yolov8_seg_baseline"))
run_dir.mkdir(parents=True, exist_ok=True)

flat = {k: v for k, v in metrics.items() if not isinstance(v, list)}
df = pd.DataFrame([flat])
csv_path = run_dir / "metrics_summary.csv"
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)

# Save per-class if available
names = cfg.get("names", None)
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]

if "seg_maps_per_class" in metrics and isinstance(metrics["seg_maps_per_class"], list):
    per = metrics["seg_maps_per_class"]
    if isinstance(names, list) and len(names) == len(per):
        dfc = pd.DataFrame({"class": names, "mAP": per})
    else:
        dfc = pd.DataFrame({"class_id": list(range(len(per))), "mAP": per})
    per_path = run_dir / "metrics_per_class.csv"
    dfc.to_csv(per_path, index=False)
    print("Saved:", per_path)


## 7) Predict on validation images (visual outputs)
Saves overlays to a folder and displays a few examples.

In [ ]:

# === Cell 7: Predictions ===
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

val_imgs = list_images(Path(cfg["val"]))
if len(val_imgs) == 0:
    raise FileNotFoundError("No validation images found to run prediction on.")

sample = random.sample(val_imgs, k=min(6, len(val_imgs)))

pred_results = model.predict(
    source=[str(p) for p in sample],
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.5,
    device=0 if torch.cuda.is_available() else "cpu",
    save=True,
    project="runs",
    name="rwh_yolov8_seg_predictions",
)

# Display saved prediction images (Ultralytics writes to runs/.../predict)
pred_dir = Path("runs") / "rwh_yolov8_seg_predictions"
# Find most recent predict folder
cands = sorted(pred_dir.parent.glob("rwh_yolov8_seg_predictions*"), key=lambda p: p.stat().st_mtime, reverse=True)
out_dir = cands[0] if cands else pred_dir
print("Prediction outputs:", out_dir)

outs = list(out_dir.glob("*.jpg")) + list(out_dir.glob("*.png"))
plt.figure(figsize=(14, 8))
for i, p in enumerate(outs[:6], 1):
    plt.subplot(2, 3, i)
    plt.imshow(Image.open(p))
    plt.title(p.name)
    plt.axis("off")
plt.tight_layout()
plt.show()


## 8) Save model weights
Copies best weights to Google Drive (optional) and zips the run folder for download.

In [ ]:

# === Cell 8: Save artifacts ===
import shutil
from pathlib import Path

# Locate best.pt from the training run
runs = Path("runs")
best_candidates = list(runs.rglob("best.pt"))
if not best_candidates:
    raise FileNotFoundError("Could not find best.pt. Check training logs.")
best_pt = max(best_candidates, key=lambda p: p.stat().st_mtime)
print("Best weights:", best_pt)

# Optional: copy to Drive
COPY_TO_DRIVE = False
DRIVE_DIR = "/content/drive/MyDrive/rwh_seg_models"
if COPY_TO_DRIVE:
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    dst = Path(DRIVE_DIR) / best_pt.name
    shutil.copy2(best_pt, dst)
    print("Copied to:", dst)

# Zip the main run directory (so you can download results)
run_root = best_pt.parent.parent  # .../weights -> run folder
zip_out = Path("/content") / f"{run_root.name}.zip"
shutil.make_archive(str(zip_out).replace(".zip",""), "zip", root_dir=run_root)
print("Zipped run folder:", zip_out)

# Offer download
from google.colab import files
files.download(str(zip_out))
